# MCQ Prompt-Format Search v2: Fixing an Ill-Posed Question

**Bug found in the first version of this search (`mcq_prompt_format_search.ipynb`):**
every prompt variant asked "What is in this image? A) cat B) dog C) fish D) bird" —
but the image is a 2x2 grid containing **four different objects at once**, and the
prompt never said *which* cell to answer about. That question has no single correct
answer from the image alone; near-chance accuracy on that version reflected an
ill-posed task, not (necessarily) a real model capability limit.

This version fixes that: every prompt variant now explicitly names the **target
cell's location** before asking which option matches what's there, using three ways
of referring to a grid location (reusing `vis_head/imagenet_grid.py` helpers):

- **word position** (`top-left`, `bottom-right`, ...) via `position_words_prompt`
- **row/col coordinate** (`(1, 1)`, `(2, 2)`, ...) via `bare_row_col_prompt`
- **pixel bounding box** (`(x0,y0)` to `(x1,y1)`) via `grid.cell_bboxes`

crossed with the same verbose-vs-short phrasing spectrum as before.

In [1]:
import sys, re
from pathlib import Path
REPO_ROOT = Path("/mnt/abka03/Projects/vis-head")
sys.path.insert(0, str(REPO_ROOT))

import torch
import numpy as np
import pandas as pd
from transformers import (AutoProcessor, LlavaForConditionalGeneration,
                           LlavaNextForConditionalGeneration, Gemma3ForConditionalGeneration)

from vis_head.imagenet_grid import (DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names,
                                     sample_grid, position_words_prompt, bare_row_col_prompt)

ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_SAMPLES = 24
SEED = 555
OPTION_LETTERS = ["A", "B", "C", "D"]
DEVICE = "cuda:0"

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")

1000 ImageNet classes available


In [2]:
def build_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    distractor_idx = rng.choice(len(other_names), size=3, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    return grid, target_cell, options, correct_letter


def parse_letter(text):
    match = re.search(r"\b([ABCD])\b", text.upper())
    return match.group(1) if match else None


sample_rng = np.random.RandomState(SEED)
FIXED_SAMPLES = [build_sample(sample_rng) for _ in range(N_SAMPLES)]
print(f"Built {len(FIXED_SAMPLES)} fixed evaluation samples")

Built 24 fixed evaluation samples


## Location-aware prompt-format candidates

Every variant now names the target cell's location before the options. Word-position
and row/col use the existing dataset helpers; bbox uses raw pixel coordinates
(the same style as the location-cue baseline in `prompt_phrasing_vis_head_vs_causal.ipynb`).

In [3]:
def loc_word(target_cell):
    return position_words_prompt(target_cell + 1, ROWS, COLS)  # e.g. "top-left"


def loc_rowcol(target_cell):
    return bare_row_col_prompt(target_cell + 1, COLS)  # e.g. "(1, 1)"


def loc_bbox(grid, target_cell):
    x0, y0, x1, y1 = grid.cell_bboxes[target_cell]
    return f"({x0}, {y0}) to ({x1}, {y1})"


PROMPT_VARIANTS = {
    "verbose_word_position": lambda grid, tc, opts: (
        f"Which of the following is shown in the {loc_word(tc)} part of this image? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + ". Answer with only the letter."
    ),
    "short_word_position": lambda grid, tc, opts: (
        f"What is in the {loc_word(tc)} picture? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + " Answer with one letter."
    ),
    "verbose_row_col": lambda grid, tc, opts: (
        f"Which of the following is shown at grid position {loc_rowcol(tc)} (row, column) "
        "in this image? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + ". Answer with only the letter."
    ),
    "short_row_col": lambda grid, tc, opts: (
        f"What is at position {loc_rowcol(tc)} in this image? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + " Answer with one letter."
    ),
    "verbose_bbox": lambda grid, tc, opts: (
        f"The object is inside the box from {loc_bbox(grid, tc)}. Which of the following "
        "is it? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + ". Answer with only the letter."
    ),
    "short_newline_word_position": lambda grid, tc, opts: (
        f"What is in the {loc_word(tc)} picture?\n"
        + "\n".join(f"{l}. {n}" for l, n in zip(OPTION_LETTERS, opts))
        + "\nAnswer:"
    ),
}
for name, fn in PROMPT_VARIANTS.items():
    print(f"--- {name} ---")
    print(fn(FIXED_SAMPLES[0][0], 0, ["cat", "dog", "fish", "bird"]))
    print()

--- verbose_word_position ---
Which of the following is shown in the top-left part of this image? A) cat  B) dog  C) fish  D) bird. Answer with only the letter.

--- short_word_position ---
What is in the top-left picture? A) cat  B) dog  C) fish  D) bird Answer with one letter.

--- verbose_row_col ---
Which of the following is shown at grid position (1, 1) (row, column) in this image? A) cat  B) dog  C) fish  D) bird. Answer with only the letter.

--- short_row_col ---
What is at position (1, 1) in this image? A) cat  B) dog  C) fish  D) bird Answer with one letter.

--- verbose_bbox ---
The object is inside the box from (0, 0) to (256, 256). Which of the following is it? A) cat  B) dog  C) fish  D) bird. Answer with only the letter.

--- short_newline_word_position ---
What is in the top-left picture?
A. cat
B. dog
C. fish
D. bird
Answer:



In [4]:
def run_prompt_search(model, processor, model_kind, use_instruct_template_text_from=None):
    template_processor = use_instruct_template_text_from or processor
    results = {}
    for variant_name, prompt_fn in PROMPT_VARIANTS.items():
        correct = 0
        unparsed = 0
        shown = []
        for grid, target_cell, options, correct_letter in FIXED_SAMPLES:
            prompt = prompt_fn(grid, target_cell, options)
            messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
            text = template_processor.apply_chat_template([messages], tokenize=False, add_generation_prompt=True)
            text = text[0] if isinstance(text, list) else text
            inputs = processor(text=text, images=[grid.grid], return_tensors="pt").to(DEVICE)
            prompt_length = int(inputs["input_ids"].shape[1])
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
            gen_ids = out[0][prompt_length:]
            gen_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
            pred = parse_letter(gen_text)
            if pred is None:
                unparsed += 1
            elif pred == correct_letter:
                correct += 1
            if len(shown) < 3:
                shown.append((correct_letter, gen_text))
        acc = correct / len(FIXED_SAMPLES)
        results[variant_name] = {"accuracy": acc, "unparsed": unparsed, "examples": shown}
        print(f"  [{model_kind}] {variant_name:26s} acc={acc:.3f}  unparsed={unparsed}/{len(FIXED_SAMPLES)}  "
              f"examples={[(c, t) for c, t in shown]}")
    return results


all_results = {}

## LLaVA-1.5-7B

In [5]:
proc = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["llava-1.5-7b"] = run_prompt_search(model, proc, "llava-1.5-7b")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

  [llava-1.5-7b] verbose_word_position      acc=0.750  unparsed=0/24  examples=[('A', 'B) missile'), ('A', 'A) speedboat'), ('C', 'C) megalith')]


  [llava-1.5-7b] short_word_position        acc=0.500  unparsed=0/24  examples=[('A', 'B'), ('A', 'A'), ('C', 'C) megalith')]


  [llava-1.5-7b] verbose_row_col            acc=0.292  unparsed=0/24  examples=[('A', 'C'), ('A', 'B'), ('C', 'D')]


  [llava-1.5-7b] short_row_col              acc=0.250  unparsed=0/24  examples=[('A', 'C'), ('A', 'B'), ('C', 'A')]


  [llava-1.5-7b] verbose_bbox               acc=0.333  unparsed=0/24  examples=[('A', 'B) missile'), ('A', 'B'), ('C', 'D')]


  [llava-1.5-7b] short_newline_word_position acc=0.542  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'A')]


## LLaVA-1.6-Vicuna-7B (instruct)

In [6]:
proc = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-vicuna-7b-hf", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["llava-1.6-vicuna-7b"] = run_prompt_search(model, proc, "llava-1.6-vicuna-7b")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

  [llava-1.6-vicuna-7b] verbose_word_position      acc=0.750  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'C')]


  [llava-1.6-vicuna-7b] short_word_position        acc=0.667  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'C')]


  [llava-1.6-vicuna-7b] verbose_row_col            acc=0.333  unparsed=0/24  examples=[('A', 'A'), ('A', 'B'), ('C', 'D')]


  [llava-1.6-vicuna-7b] short_row_col              acc=0.292  unparsed=0/24  examples=[('A', 'A'), ('A', 'B'), ('C', 'C')]


  [llava-1.6-vicuna-7b] verbose_bbox               acc=0.292  unparsed=0/24  examples=[('A', 'B'), ('A', 'B'), ('C', 'D')]


  [llava-1.6-vicuna-7b] short_newline_word_position acc=0.708  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'C')]


## Gemma-3-4B-it (instruct)

Tested first since it has its own working chat template; base model reuses this
template's rendered text below (base has no chat template of its own).

In [7]:
gemma_it_proc = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
model = Gemma3ForConditionalGeneration.from_pretrained(
    "google/gemma-3-4b-it", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["gemma-3-4b-it"] = run_prompt_search(model, gemma_it_proc, "gemma-3-4b-it")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

  [gemma-3-4b-it] verbose_word_position      acc=0.625  unparsed=0/24  examples=[('A', 'B'), ('A', 'A'), ('C', 'C')]


  [gemma-3-4b-it] short_word_position        acc=0.667  unparsed=0/24  examples=[('A', 'B'), ('A', 'A'), ('C', 'C')]


  [gemma-3-4b-it] verbose_row_col            acc=0.542  unparsed=0/24  examples=[('A', 'A'), ('A', 'B'), ('C', 'C')]


  [gemma-3-4b-it] short_row_col              acc=0.417  unparsed=0/24  examples=[('A', 'A'), ('A', 'B'), ('C', 'C')]


  [gemma-3-4b-it] verbose_bbox               acc=0.375  unparsed=0/24  examples=[('A', 'C'), ('A', 'A'), ('C', 'C')]


  [gemma-3-4b-it] short_newline_word_position acc=0.667  unparsed=0/24  examples=[('A', 'The correct answer is **B. missile'), ('A', 'The correct answer is **A. speedboat'), ('C', 'The correct answer is **C. megal')]


## Gemma-3-4B-pt (base)

In [8]:
gemma_pt_proc = AutoProcessor.from_pretrained("google/gemma-3-4b-pt")
model = Gemma3ForConditionalGeneration.from_pretrained(
    "google/gemma-3-4b-pt", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["gemma-3-4b-pt"] = run_prompt_search(
    model, gemma_pt_proc, "gemma-3-4b-pt", use_instruct_template_text_from=gemma_it_proc)
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

  [gemma-3-4b-pt] verbose_word_position      acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhich of the following is shown'), ('A', '\n\nWhich of the following is shown'), ('C', '\n\nWhich of the following is shown')]


  [gemma-3-4b-pt] short_word_position        acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhat is the model of the'), ('A', '\n\nWhat is the name of the'), ('C', '\n\nWhat is the model of the')]


  [gemma-3-4b-pt] verbose_row_col            acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhich of the following is shown'), ('A', '\n\nWhich of the following is shown'), ('C', '\n\nWhich of the following is shown')]


  [gemma-3-4b-pt] short_row_col              acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhat is the model of the'), ('A', '\n\nWhat is the name of the'), ('C', '\n\nWhat is the name of the')]


  [gemma-3-4b-pt] verbose_bbox               acc=0.000  unparsed=24/24  examples=[('A', '\n\nThe object is inside the box'), ('A', '\n\nThe object is inside the box'), ('C', '\n\nThe object is inside the box')]


  [gemma-3-4b-pt] short_newline_word_position acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhat is in the bottom-'), ('A', '\n\nWhat is in the bottom-'), ('C', '\n\nWhat is the name of the')]


## Summary table

In [9]:
rows = []
for model_kind, variant_results in all_results.items():
    for variant_name, r in variant_results.items():
        rows.append({"model": model_kind, "prompt_variant": variant_name,
                     "accuracy": r["accuracy"], "unparsed": r["unparsed"]})
df = pd.DataFrame(rows)
pivot = df.pivot(index="prompt_variant", columns="model", values="accuracy")
pivot = pivot.reindex(list(PROMPT_VARIANTS.keys()))
pd.set_option("display.width", 140)
print("Accuracy by prompt variant x model (chance = 0.25):\n")
print(pivot.round(3).to_string())

print("\nVariants where ALL models beat chance (>0.25):")
above_chance_all = pivot[(pivot > 0.25).all(axis=1)]
print(above_chance_all.round(3).to_string() if len(above_chance_all) else "  (none)")

print("\nMean accuracy per variant across models:")
print(pivot.mean(axis=1).round(3).sort_values(ascending=False).to_string())

Accuracy by prompt variant x model (chance = 0.25):

model                        gemma-3-4b-it  gemma-3-4b-pt  llava-1.5-7b  llava-1.6-vicuna-7b
prompt_variant                                                                              
verbose_word_position                0.625            0.0         0.750                0.750
short_word_position                  0.667            0.0         0.500                0.667
verbose_row_col                      0.542            0.0         0.292                0.333
short_row_col                        0.417            0.0         0.250                0.292
verbose_bbox                         0.375            0.0         0.333                0.292
short_newline_word_position          0.667            0.0         0.542                0.708

Variants where ALL models beat chance (>0.25):
  (none)

Mean accuracy per variant across models:
prompt_variant
verbose_word_position          0.531
short_newline_word_position    0.479
short_word_posi

## Result

(filled in after running)